# Module 1 — Corpus Preparation

Load BnSentMix, inspect the corpus, normalize Bangla-English text, and create one reproducible 70/15/15 split for every model.

In [ ]:
from collections import Counter
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import LABEL_ID_TO_NAME, PROCESSED_DATA_PATH, RAW_DATA_PATH
from src.dataset_utils import create_fixed_data_splits
from src.preprocessing import preprocess_text

## Inspect the raw corpus

In [ ]:
raw_data = pd.read_csv(RAW_DATA_PATH)
print('Shape:', raw_data.shape)
print('Columns:', raw_data.columns.tolist())
print('Missing values:\n', raw_data.isna().sum())
print('Exact duplicate rows:', raw_data.duplicated().sum())
print('Label counts:\n', raw_data['Label'].value_counts().sort_index())
raw_data.head(10)

## Verify the selected cleaning rules

The reusable function preserves Bangla, Romanized Bangla, and negation words. English-only stemming, lemmatization, stop-word removal, and spelling correction are intentionally excluded.

In [ ]:
examples = [
    'Movie ta REALLY bhalo!!! https://example.com',
    'movie ta bhalo na',
    'সার্ভিস ভালো না',
]
for example in examples:
    print(example, '->', preprocess_text(example))

## Prepare and save the complete corpus

In [ ]:
required_columns = {'Sentence', 'Label'}
missing_columns = required_columns.difference(raw_data.columns)
if missing_columns:
    raise ValueError(f'Missing columns: {sorted(missing_columns)}')

data = raw_data[['Sentence', 'Label']].copy()
data['Sentence'] = data['Sentence'].astype('string').str.strip()
data = data.dropna(subset=['Sentence', 'Label'])
data = data[data['Sentence'] != '']
data = data.drop_duplicates(subset=['Sentence', 'Label']).reset_index(drop=True)

data['label_id'] = pd.to_numeric(data['Label'], errors='coerce')
data = data.dropna(subset=['label_id'])
data['label_id'] = data['label_id'].astype(int)
unknown_labels = set(data['label_id']).difference(LABEL_ID_TO_NAME)
if unknown_labels:
    raise ValueError(f'Unknown labels: {sorted(unknown_labels)}')

data = data.rename(columns={'Sentence': 'original_text'})
data['label'] = data['label_id'].map(LABEL_ID_TO_NAME)
data['processed_text'] = data['original_text'].map(preprocess_text)
data = data[data['processed_text'] != ''].reset_index(drop=True)
data['token_count'] = data['processed_text'].str.split().str.len()
processed_data = data[['original_text', 'processed_text', 'label', 'label_id', 'token_count']]

# Avoid rewriting the large file when preprocessing produced the same data.
should_save = True
if PROCESSED_DATA_PATH.exists():
    saved_data = pd.read_csv(PROCESSED_DATA_PATH)
    same_columns = list(saved_data.columns) == list(processed_data.columns)
    same_values = same_columns and saved_data.astype(str).equals(processed_data.astype(str))
    should_save = not same_values

if should_save:
    processed_data.to_csv(PROCESSED_DATA_PATH, index=False)
    print(f'Saved processed data to: {PROCESSED_DATA_PATH}')
else:
    print('Processed data is unchanged; the existing CSV was kept.')
train_data, validation_data, test_data = create_fixed_data_splits(processed_data)
print(f'Processed samples: {len(processed_data):,}')
print(f'Splits: train={len(train_data):,}, validation={len(validation_data):,}, test={len(test_data):,}')

## Review corpus statistics

In [ ]:
display(processed_data.groupby('label', sort=False).head(3)[['original_text', 'processed_text', 'label']])
display(processed_data['token_count'].describe(percentiles=[0.90, 0.95]))
display(processed_data['label'].value_counts())
token_counts = Counter(token for text in processed_data['processed_text'] for token in text.split())
display(pd.DataFrame(token_counts.most_common(20), columns=['token', 'frequency']))